## Clean WDM Retrieval Notebook

# This notebook is a thin Jupyter wrapper around `wdm_retrieval_clean.py`.

#It keeps the retrieval logic clean and avoids the earlier `numpy.ndarray` issue with `wdm.csvtowdm` by ensuring all WDM inputs remain pandas objects with a `DatetimeIndex`.

#**Before running this notebook:** make sure your Earthdata credentials and token setup are already working in your environment.

In [2]:
import requests
import io
import pandas
import numpy as np
#import matplotlib.pyplot as plt
from subprocess import Popen
import platform
import os
import shutil
from getpass import getpass
import netrc
from base64 import b64encode
#import earthaccess

# Establish files required for API Access
# Set homeDir to current working directory
homeDir = os.path.expanduser("~/")
#homeDir = os.getcwd() + os.sep
# Create .urs_cookies and .dodsrc files in current directory
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))

print('Saved .urs_cookies and .dodsrc to:', homeDir)

In [7]:
# Establish files required for API Access
# Set homeDir to current working directory
homeDir = os.path.expanduser("~/")
#homeDir = os.getcwd() + os.sep
# Create .urs_cookies and .dodsrc files in current directory
with open(homeDir + '.urs_cookies', 'w') as file:
    file.write('')
with open(homeDir + '.dodsrc', 'w') as file:
    file.write('HTTP.COOKIEJAR={}.urs_cookies\n'.format(homeDir))
    file.write('HTTP.NETRC={}.netrc'.format(homeDir))

print('Saved .urs_cookies and .dodsrc to:', homeDir)
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()

print('Saved .netrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)
    # Earthdata Login URL for obtaining the token, and creating one if it doesn't exist
url = 'https://urs.earthdata.nasa.gov/api/users/find_or_create_token'

# Earthdata Login credential prompts
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

# Get credentials from user input
auth = netrc.netrc(file=homeDir + '.netrc')
username, _, password = auth.authenticators('urs.earthdata.nasa.gov')

# Encode credentials using Base64
credentials = b64encode(f"{username}:{password}".encode('utf-8')).decode('utf-8')

# Headers with the Basic Authorization
headers = {
    'Authorization': f'Basic {credentials}'
}

# Make the POST request to get the token
response = requests.post(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON to get the token
    token_info = response.json()
    token = token_info.get("access_token")
    print("Token retrieved successfully")

    # Define the path for the .edl_token file in the home directory
    token_file_path = homeDir + ".edl_token"

    # Write the token to the .edl_token file
    with open(token_file_path, 'w') as token_file:
        token_file.write(token)

    print(f"Token saved to {token_file_path}")

else:
    print("Failed to retrieve token:", response.text)

Saved .urs_cookies and .dodsrc to: C:\Users\KVENABLE/


In [24]:
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL to call for authentication
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

with open(homeDir + '.netrc', 'w') as file:
    file.write('machine {} login {} password {}'.format(urs, getpass(prompt=prompts[0]), getpass(prompt=prompts[1])))
    file.close()

print('Saved .netrc to:', homeDir)

# Set appropriate permissions for Linux/macOS
if platform.system() != "Windows":
    Popen('chmod og-rw ~/.netrc', shell=True)

Saved .netrc to: C:\Users\KVENABLE/


In [25]:
# Earthdata Login URL for obtaining the token, and creating one if it doesn't exist
url = 'https://urs.earthdata.nasa.gov/api/users/find_or_create_token'

# Earthdata Login credential prompts
prompts = ['Enter NASA Earthdata Login Username \n(or create an account at urs.earthdata.nasa.gov): ',
           'Enter NASA Earthdata Login Password: ']

# Get credentials from user input
auth = netrc.netrc(file=homeDir + '.netrc')
username, _, password = auth.authenticators('urs.earthdata.nasa.gov')

# Encode credentials using Base64
credentials = b64encode(f"{username}:{password}".encode('utf-8')).decode('utf-8')

# Headers with the Basic Authorization
headers = {
    'Authorization': f'Basic {credentials}'
}

# Make the POST request to get the token
response = requests.post(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON to get the token
    token_info = response.json()
    token = token_info.get("access_token")
    print("Token retrieved successfully")

    # Define the path for the .edl_token file in the home directory
    token_file_path = homeDir + ".edl_token"

    # Write the token to the .edl_token file
    with open(token_file_path, 'w') as token_file:
        token_file.write(token)

    print(f"Token saved to {token_file_path}")

else:
    print("Failed to retrieve token:", response.text)

Token retrieved successfully
Token saved to C:\Users\KVENABLE/.edl_token


In [ ]:
import importlib
import os
import pandas as pd
from pathlib import Path

import wdm_retrieval_clean as wrc

importlib.reload(wrc)
print("Loaded from:", wrc.__file__)

## Edit these parameters

#You can point `station_csv` to your own station list, or keep the provided example file.

In [34]:
import pandas as pd

workspace = Path.cwd()
station_csv = workspace / "station_example.csv"
start_date = "2023-01-01"
end_date = "2023-12-31"
wdm_file = workspace / "MetData_clean_notebook_042226b.wdm"
log_file = workspace / "MetLog_clean_notebook_042226b.txt"
print(f"Workspace: {workspace}")
print(f"Station CSV: {station_csv}")
print(f"Date range: {start_date} to {end_date}")
print(f"WDM output: {wdm_file}")
print(f"Log output: {log_file}")

Workspace: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS
Station CSV: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\station_example.csv
Date range: 2023-01-01 to 2023-12-31
WDM output: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\MetData_clean_notebook_042226b.wdm
Log output: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\MetLog_clean_notebook_042226b.txt


## Preview stations and dataset selection
##"This cell validates the station file and shows whether each station will use NLDAS or GLDAS."

In [18]:
stations = wrc.load_stations(station_csv)
preview_rows = []
for station in stations:
    catalog, timestep_hours, dataset_name = wrc.get_catalog(station)
    preview_rows.append({
        "StationName": station.name,
        "Latitude": station.latitude,
        "Longitude": station.longitude,
        "TimeZoneAdjustment": station.timezone_adjustment,
        "Dataset": dataset_name,
        "TimeStepHours": timestep_hours,
    })
pd.DataFrame(preview_rows)

,StationName,Latitude,Longitude,TimeZoneAdjustment,Dataset,TimeStepHours
0,CentralPark,40.785091,-73.968285,-5.0,NLDAS,1
1,GoldenGate,37.819929,-122.478255,-8.0,NLDAS,1
2,MillenniumPark,41.882702,-87.619392,-6.0,NLDAS,1


## Constituents to be written
##These are the constituents handled by the clean workflow

In [35]:
wrc.CONSTITUENTS

['ATEMP', 'PRECIP', 'SOLR', 'WIND', 'CLOU', 'DEWP', 'ATMP']

## Preview DEWP calculation
##This cell fetches and previews the computed dew point timeseries for the first station before the full WDM run.

In [29]:
wrc.ensure_earthdata_credentials()

if not stations:
    raise ValueError("No stations were loaded from the station CSV.")

dewp_preview_result = wrc.build_result_for_constituent(
    station=stations[0],
    constituent="DEWP",
    start_date=start_date,
    end_date=end_date,
)

print(f"Preview station: {stations[0].name}")
print(f"DEWP source variables: {dewp_preview_result.variable_name}")
dewp_preview_result.series.to_frame(name="DEWP").head()

Loaded from: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\wdm_retrieval_clean.py
Constituents: ['ATEMP', 'PRECIP', 'SOLR', 'WIND', 'CLOU', 'DEWP', 'ATMP']


## Run retrieval and write the WDM file
##"This cell will call the clean retrieval pipeline, build the WDM file, and create the MetLog text file.

In [36]:
created_wdm, created_log = wrc.run_retrieval(stations=stations, start_date=start_date, end_date=end_date, wdm_path=wdm_file, log_path=log_file)
print(f"Created WDM file: {created_wdm}")
print(f"Created log file: {created_log}")

Created WDM file: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\MetData_clean_notebook_042226b.wdm
Created log file: C:\Users\KVENABLE\NLDASV2_Giovanni_BASINS\MetLog_clean_notebook_042226b.txt


## Quick output check

In [33]:
print("WDM exists:", Path(created_wdm).exists())
print("Log exists:", Path(created_log).exists())

if Path(created_log).exists():
    with open(created_log, "r", encoding="utf-8") as f:
        log_preview = f.readlines()[:15]
    print("".join(log_preview))

WDM exists: True
Log exists: True
Started downloading data at 2026-04-22T12:54:14.139587 and saving in MetData_clean_notebook_042226a.wdm
Station: CentralPark, Latitude: 40.785091, Longitude: -73.968285, TimeZoneAdjustment: -5.0
Constituent: ATEMP, Column Name: ATEMP, DSN: 1
Constituent: PRECIP, Column Name: PRECIP, DSN: 2
Constituent: SOLR, Column Name: SOLR, DSN: 3
Constituent: WIND, Column Name: WIND, DSN: 4
Constituent: CLOU, Column Name: CLOU, DSN: 5
Constituent: DEWP, Column Name: DEWP, DSN: 6
Constituent: ATMP, Column Name: ATMP, DSN: 7
Station: GoldenGate, Latitude: 37.819929, Longitude: -122.478255, TimeZoneAdjustment: -8.0
Constituent: ATEMP, Column Name: ATEMP, DSN: 8
Constituent: PRECIP, Column Name: PRECIP, DSN: 9
Constituent: SOLR, Column Name: SOLR, DSN: 10
Constituent: WIND, Column Name: WIND, DSN: 11
Constituent: CLOU, Column Name: CLOU, DSN: 12

